In [1]:
# Importar librerias
import pandas as pd

from funciones import *

In [2]:
# Generar lista de tickers del Fichero constituents del S&P 500
data_folder = "../data"
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

503

In [3]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector","GICS Sub-Industry", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "SubIndustry", 
    "Date added": "DateAdded"
    }, inplace=True)

In [4]:
df_tickers.head()

,Ticker,Sector,SubIndustry,DateAdded
0,MMM,Industrials,Industrial Conglomerates,1957-03-04
1,AOS,Industrials,Building Products,2017-07-26
2,ABT,Health Care,Health Care Equipment,1957-03-04
3,ABBV,Health Care,Biotechnology,2012-12-31
4,ACN,Information Technology,IT Consulting & Other Services,2011-07-06


In [5]:
df_prices = extraer_precios(tickers_list)

In [6]:
df_index = extraer_precios(['SPY'])
df_betas = calcular_betas(df_prices, df_index)
df_betas.head()

c:\Users\sebas\OneDrive\Documentos\python-development\FinDataMining\notebooks\funciones.py:160: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df_retornos = df_pivot.pct_change()


,Fecha,Ticker,Beta
0,2022-06-06,A,NaN
1,2022-06-13,A,NaN
2,2022-06-20,A,NaN
3,2022-06-27,A,NaN
4,2022-07-04,A,NaN


In [7]:
df_betas.tail()

,Fecha,Ticker,Beta
105122,2026-05-04,ZTS,0.6834
105123,2026-05-11,ZTS,0.6920
105124,2026-05-18,ZTS,0.7862
105125,2026-05-25,ZTS,0.7242
105126,2026-06-01,ZTS,0.5988


In [8]:
df_prices.head()

,Fecha,Close,Ticker
0,2022-06-06,99.760010,MMM
1,2022-06-13,94.099823,MMM
2,2022-06-20,97.353897,MMM
3,2022-06-27,93.121422,MMM
4,2022-07-04,93.288132,MMM


In [9]:
df_index.head()

,Fecha,Close,Capital Gains,Ticker
0,2022-06-06,368.857422,0.0,SPY
1,2022-06-13,346.203705,0.0,SPY
2,2022-06-20,370.716919,0.0,SPY
3,2022-06-27,362.315765,0.0,SPY
4,2022-07-04,369.376984,0.0,SPY


In [10]:
df_fundamentals = extraer_datos_fundamentales(tickers_list)

In [11]:
df_fundamentals.head()

,Fecha,Total Revenue,Gross Profit,Operating Income,Net Income,EBITDA,Basic Average Shares,Cash And Cash Equivalents,Current Debt,Long Term Debt,Total Debt,Stockholders Equity,Total Assets,Current Assets,Current Liabilities,Ticker
0,2025-12-31,2.494800e+10,9.957000e+09,4.791000e+09,3.250000e+09,6.467000e+09,537400000.0,5.235000e+09,1.670000e+09,1.093200e+10,1.260200e+10,4.702000e+09,3.773300e+10,1.638700e+10,9.595000e+09,MMM
1,2024-12-31,2.457500e+10,1.012800e+10,4.822000e+09,4.173000e+09,7.373000e+09,550800000.0,5.600000e+09,1.919000e+09,1.112500e+10,1.304400e+10,3.842000e+09,3.986800e+10,1.588400e+10,1.125600e+10,MMM
2,2023-12-31,2.461000e+10,9.627000e+09,-1.072500e+10,-6.995000e+09,-8.343000e+09,553900000.0,5.735000e+09,2.947000e+09,1.308800e+10,1.675100e+10,4.807000e+09,5.058000e+10,1.637900e+10,1.529700e+10,MMM
3,2022-12-31,2.616100e+10,1.030800e+10,1.916000e+09,5.777000e+09,6.497000e+09,566000000.0,3.655000e+09,1.938000e+09,1.400100e+10,1.685500e+10,1.472200e+10,4.645500e+10,1.468800e+10,9.523000e+09,MMM
4,2021-12-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,MMM


In [12]:
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# --- LA CORRECCIÓN DEL LOOKAHEAD BIAS ---
# Desplazamos la fecha del reporte 60 días hacia el futuro para simular 
# el momento en que el dato se vuelve realmente público para los inversores.
retraso_dias = 60
df_fundamentals['Fecha'] = df_fundamentals['Fecha'] + pd.DateOffset(days=retraso_dias)
# ----------------------------------------

# 2. Hacer el OUTER merge
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# 3. Ordenar cronológicamente (CRUCIAL para el ffill)
df_merged = df_merged.sort_values(by=['Ticker', 'Fecha'])

# 4. Aplicar Forward Fill
# Identificamos qué columnas vienen de los estados financieros (las que tienen NaN)
cols_financieras = ['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 
                    'EBITDA', 'Basic Average Shares', 'Cash And Cash Equivalents', 'Current Debt', 
                    'Long Term Debt', 'Total Debt', 'Stockholders Equity',
                    'Total Assets', 'Current Assets', 'Current Liabilities'
                    ]

# Aplicamos el ffill() EXCLUSIVAMENTE a esas columnas, agrupando por Ticker
df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

# 5. Limpiar las filas huérfanas creadas por el outer merge
df_merged_clean = df_merged.dropna(subset=['Close'])

# 6. ELIMINAR LOS NaN INICIALES 
# Filtramos por una columna fundamental crítica. Las filas anteriores al primer
# reporte financiero disponible serán eliminadas de forma limpia.
columna_critica = 'EBITDA' # O 'Net Income', la que uses sí o sí en tus ratios
df_merged_clean = df_merged_clean.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los betas calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_betas, on=['Ticker', 'Fecha'], how='left')

print(df_merged_clean.head())

       Fecha       Close Ticker  Total Revenue  Gross Profit  \
0 2023-01-02  144.103073      A   6.848000e+09  3.722000e+09   
1 2023-01-09  153.129654      A   6.848000e+09  3.722000e+09   
2 2023-01-16  152.153778      A   6.848000e+09  3.722000e+09   
3 2023-01-23  151.929321      A   6.848000e+09  3.722000e+09   
4 2023-01-30  150.816910      A   6.848000e+09  3.722000e+09   

   Operating Income    Net Income        EBITDA  Basic Average Shares  \
0      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
1      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
2      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
3      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   
4      1.618000e+09  1.254000e+09  1.905000e+09           299000000.0   

   Cash And Cash Equivalents  Current Debt  Long Term Debt    Total Debt  \
0               1.053000e+09    36000000.0    2.733000e+09  2.769000e+09   
1               1.053000

In [13]:
df_merged_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78535 entries, 0 to 78534
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Fecha                      78535 non-null  datetime64[ns]
 1   Close                      78535 non-null  float64       
 2   Ticker                     78535 non-null  object        
 3   Total Revenue              78496 non-null  float64       
 4   Gross Profit               77411 non-null  float64       
 5   Operating Income           78535 non-null  float64       
 6   Net Income                 78326 non-null  float64       
 7   EBITDA                     78535 non-null  float64       
 8   Basic Average Shares       78256 non-null  float64       
 9   Cash And Cash Equivalents  78287 non-null  float64       
 10  Current Debt               67363 non-null  float64       
 11  Long Term Debt             74754 non-null  float64       
 12  Tota

In [14]:
df_with_metrics = calcular_metricas(df_merged_clean)


In [15]:
# Luego de calcular los ratios, se eliminan las columnas financieras 
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78535 entries, 0 to 78534
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               78535 non-null  datetime64[ns]
 1   Close               78535 non-null  float64       
 2   Ticker              78535 non-null  object        
 3   Beta                70795 non-null  float64       
 4   MarketCap           78256 non-null  float64       
 5   EnterpriseValue     78256 non-null  float64       
 6   PE_Trailing         78086 non-null  float64       
 7   EnterpriseToEbitda  78256 non-null  float64       
 8   PriceToBook         78047 non-null  float64       
 9   operatingMargins    78496 non-null  float64       
 10  profitMargins       78326 non-null  float64       
 11  returnOnEquity      78117 non-null  float64       
 12  ReturnOnAssets      78287 non-null  float64       
 13  debtToEquity        78287 non-null  float64   

In [16]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78535 entries, 0 to 78534
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               78535 non-null  datetime64[ns]
 1   Close               78535 non-null  float64       
 2   Ticker              78535 non-null  object        
 3   Beta                70795 non-null  float64       
 4   MarketCap           78256 non-null  float64       
 5   EnterpriseValue     78256 non-null  float64       
 6   PE_Trailing         78086 non-null  float64       
 7   EnterpriseToEbitda  78256 non-null  float64       
 8   PriceToBook         78047 non-null  float64       
 9   operatingMargins    78496 non-null  float64       
 10  profitMargins       78326 non-null  float64       
 11  returnOnEquity      78117 non-null  float64       
 12  ReturnOnAssets      78287 non-null  float64       
 13  debtToEquity        78287 non-null  float64   

In [17]:
# Guardar datos extraidos en fichero raw_data
df_final.to_parquet(f"{data_folder}/raw_data.parquet")